In [2]:
using StructuralEquationModels

# Define variables

In [3]:
# five observed time points
obs = [:y1, :y2, :y3, :y4, :y5]

# intercept and slope (latent)
lat = [:i, :s]

2-element Vector{Symbol}:
 :i
 :s

# Define Graph for SEM

In [4]:
graph = @StenoGraph begin
# Intercept factor: all loadings fixed to 1
i → fixed(1)*y1 + fixed(1)*y2 + fixed(1)*y3 + fixed(1)*y4

# Slope factor: linear time scores
s → fixed(0)*y1 + fixed(1)*y2 + fixed(2)*y3 + fixed(3)*y4

# Residual variances (free) and latent (co)variances (free)
_(obs) ↔ _(obs) # variances for observed variables
_(lat) ⇔ _(lat) # variances + covariance for latent factors

# Latent means (estimate μ_i and μ_s);
# observed means fixed to 0 by omission
Symbol(1) → i + s
end

i → y1 * Fixed{Tuple{Int64}}((1,))
i → y2 * Fixed{Tuple{Int64}}((1,))
i → y3 * Fixed{Tuple{Int64}}((1,))
i → y4 * Fixed{Tuple{Int64}}((1,))
s → y1 * Fixed{Tuple{Int64}}((0,))
s → y2 * Fixed{Tuple{Int64}}((1,))
s → y3 * Fixed{Tuple{Int64}}((2,))
s → y4 * Fixed{Tuple{Int64}}((3,))
y1 ↔ y1 * nothing
y2 ↔ y2 * nothing
y3 ↔ y3 * nothing
y4 ↔ y4 * nothing
y5 ↔ y5 * nothing
i ↔ i * nothing
i ↔ s * nothing
s ↔ s * nothing
1 → i * nothing
1 → s * nothing


# Load Data

In [9]:
using CSV, DataFrames
data = CSV.read("data/lgcm.csv", DataFrames.DataFrame;
# ensures column names are valid Julia symbols
normalizenames = true,
# expect comma-separated values
delim = ',',
# handle missing data
missingstring = ["", "NA", "NaN"],
# ignore duplicate column delimiters
ignorerepeated = true
)

Row,y1,y2,y3,y4,y5,ID
,Float64,Float64,Float64,Float64,Float64,Int64
1,6.19645,9.18878,13.4271,12.4402,13.194,1
2,8.3542,11.9871,14.478,13.3899,14.3931,2
3,4.94226,11.0707,9.00444,19.2635,17.5656,3
4,9.94324,9.59148,8.9532,14.3082,14.5486,4
5,9.29546,10.2234,14.0224,13.2766,13.9788,5
6,11.8576,13.1334,18.707,13.9405,15.9113,6
7,7.28374,11.081,16.2504,13.3312,20.9018,7
8,7.83101,9.49231,10.8783,15.8569,14.5645,8
9,9.27376,9.13835,15.0126,13.1164,14.8447,9


# Create Model and Fit

In [6]:
partable = ParameterTable(
    graph,
    latent_vars   = lat,
    observed_vars = obs
)

model = Sem(
    specification = partable,
    data          = data
)
fitted = fit(model)

Fitted Structural Equation Model 
--------------------- Model ------------------- 

Structural Equation Model 
- Loss Functions 
   SemML
- Fields 
   observed:    SemObservedData 
   implied:     RAM 

------------- Optimization result ------------- 

 * Status: success

 * Candidate solution
    Final objective value:     1.354715e+01

 * Found with
    Algorithm:     L-BFGS

 * Convergence measures
    |x - x'|               = 1.51e-04 ≰ 1.5e-08
    |x - x'|/|x'|          = 2.31e-05 ≰ 0.0e+00
    |f(x) - f(x')|         = 6.73e-10 ≰ 0.0e+00
    |f(x) - f(x')|/|f(x')| = 4.97e-11 ≤ 1.0e-10
    |g(x)|                 = 1.40e-05 ≰ 1.0e-08

 * Work counters
    Seconds run:   0  (vs limit Inf)
    Iterations:    21
    f(x) calls:    69
    ∇f(x) calls:   69


In [7]:
fit_measures(fitted)

Dict{Symbol, Union{Missing, Float64}} with 8 entries:
  :minus2ll => 11368.3
  :AIC      => 11388.3
  :BIC      => 11430.4
  :χ²       => 161.662
  :dof      => 5.0
  :p_value  => 0.0
  :nparams  => 10.0
  :RMSEA    => 0.250329

In [8]:
update_estimate!(partable, fitted)
details(partable)


---------------------------------- Variables --------------------------------- 

Latent variables:    i s 
Observed variables:  y1 y2 y3 y4 y5 

---------------------------- Parameter Estimates ----------------------------- 

Loadings: 

i

  to   estimate   value_fixed   start   free   from   label   relation

  y1   1.0        1.0                   0.0    i      const   →
  y2   1.0        1.0                   0.0    i      const   →
  y3   1.0        1.0                   0.0    i      const   →
  y4   1.0        1.0                   0.0    i      const   →

s

  to   estimate   value_fixed   start   free   from   label   relation

  y1   0.0        0.0                   0.0    s      const   →
  y2   1.0        1.0                   0.0    s      const   →
  y3   2.0        2.0                   0.0    s      const   →
  y4   3.0        3.0                   0.0    s      const   →

Directed Effects: 

  from       to   estimate   value_fixed   start   free   label


Variances: 